# Recitation 3

Use the cells below for your calculations and notes.

**Measurement record**:
- 4 byte id of the measurement 
- 8 byte encoding of the GPs
- 8 bytes encoding of error/device 
- 8 bytes for timestamp 

- **every minute** positions are collect from every device 

**disk-block**
- 8KB storage
- 64-bit address

**B+tree vertices** fit into exactly one disk block and have 64bit addresses
- 64-bit addresses (8 bytes per pointer)

we cluster on time stamps and build a sparse index on time stamps 

Assume timestamps for a primary key, **10 000 people for 100days** 

blocking factor: how many records fits in a block 



In [1]:
SIZE_ID_MEASUREMENT = 4
SIZE_ENCODING_GPS = 8
SIZE_ENCODING_ERROR = 8
SIZE_TIMESTAMP = 8

SIZE_PTR = 8

SIZE_DISK_BLOCK = 8 * pow(2,10)




1. For the measurement data, what is the blocking factor for a B+ tree leaf node with forward and
backward pointer to next/previous blocks

In [2]:
# Answer
block_size = SIZE_DISK_BLOCK
total_bytes_in_block = block_size - 8 * 2 #taking to account for the forward and backward pointer 

record_size = SIZE_ID_MEASUREMENT
record_size += SIZE_ENCODING_GPS
record_size += SIZE_ENCODING_ERROR
record_size += SIZE_TIMESTAMP

SIZE_RECORD = record_size

blocking_factor = total_bytes_in_block / record_size
print("the blocking factor is: "+str(blocking_factor))

the blocking factor is: 292.0


2. What is the order of an internal B+tree vertex?

notes: internal are the none leaf nodes

In [3]:
Page_size = SIZE_DISK_BLOCK
Key_size = SIZE_TIMESTAMP
Ptr_size = SIZE_PTR

import math
def internal_n_per_vertices(Page_size, Key_size, Ptr_size):
    numerator = Page_size + Key_size
    denominator = Ptr_size + Key_size 
    print("calculating internal ptr per vertices")
    print("(Page_size + Key_size)"+"/"+"(Ptr_size + Key_size) = (" +str(Page_size)+" + "+str(Key_size)+")/("+str(Ptr_size)+" + "+str(Key_size)+")" )

    ans = float(math.floor(numerator / denominator))
    print(f"internal pointers per vertices are: {ans}\n")
    
    return ans 

maximun_n_internal_vertices = internal_n_per_vertices(Page_size,SIZE_TIMESTAMP,SIZE_PTR)


print("the answer is: "+str(maximun_n_internal_vertices))


calculating internal ptr per vertices
(Page_size + Key_size)/(Ptr_size + Key_size) = (8192 + 8)/(8 + 8)
internal pointers per vertices are: 512.0

the answer is: 512.0


3. How many measurement records are collected and how many blocks required assuming full data record
blocks?

In [4]:
def amount_of_records(update_rate_per_min, duration_in_days, amount_of_people):
    return update_rate_per_min * duration_in_days * 24 * 60 * amount_of_people


records = amount_of_records(1,100, 10000)
print("the amount of records are: "+ str(records))

import math
external_blocks = records / blocking_factor

external_blocks = math.ceil(external_blocks)

print("the amount of external blocks are: "+str(external_blocks))

def get_internal_blocks(external_blocks,maximum_n_per_vertices):
    internal_vertices = 0
    temp = external_blocks
    
    while temp != 1:
        temp = math.ceil(temp/maximum_n_per_vertices)
        internal_vertices += temp

    return internal_vertices

internal_blocks = get_internal_blocks(external_blocks,maximun_n_internal_vertices)

        


print("the amount of internal blocks are: "+str(internal_blocks))

print("the total amount of blocks are: "+str(internal_blocks + external_blocks))


the amount of records are: 1440000000
the amount of external blocks are: 4931507
the amount of internal blocks are: 9652
the total amount of blocks are: 4941159


4. What is the maximum p ossible height of a primary B+tree built over time stamps assuming full data
record blocks? (The leaf vertices are maximally packed while the interior vertices are as empty as
possible)


In [5]:
# just do log it 

# height/level is counted wihouth root
import math

def get_height(number_of_records, records_per_leaf_vertices,
               pointers_per_internal_vertices, pointers_per_root_vertices):
    numerator = number_of_records
    denominator = (pointers_per_root_vertices + 1) * records_per_leaf_vertices

    height = math.log(numerator / denominator, pointers_per_internal_vertices)

    print("Calculating height")
    print(
        "log base pointers_per_internal_vertices "
        "(number_of_records / ((pointers_per_root_vertices + 1) "
        "* records_per_leaf_vertices))"
    )
    print(
        f"= log base {pointers_per_internal_vertices} "
        f"({number_of_records} / (({pointers_per_root_vertices} + 1) "
        f"* {records_per_leaf_vertices}))"
    )
    print(
        f"= log base {pointers_per_internal_vertices} "
        f"({numerator / denominator}) = {height}"
    )
    print(f"Rounded up: ceil({height}) = {math.ceil(height)}")

    return math.ceil(height) #there can't be half levels so it is rounded up 

# internal vertices have on fewer key than pointers, so that why you floor it for the -1 to count 
# ceiling it to make it so it has at least over half

height_without_root = get_height(records, math.ceil(blocking_factor/2),math.floor((maximun_n_internal_vertices-1)/2),1)

print("height without root: "+str(math.ceil(height_without_root)))


Calculating height
log base pointers_per_internal_vertices (number_of_records / ((pointers_per_root_vertices + 1) * records_per_leaf_vertices))
= log base 255 (1440000000 / ((1 + 1) * 146))
= log base 255 (4931506.849315069) = 2.7811626396530524
Rounded up: ceil(2.7811626396530524) = 3
height without root: 3


5. For 4 above, what is the minimum height (interior nodes maximally packed as well as leaf nodes)

In [6]:
height_without_root = get_height(records, blocking_factor,maximun_n_internal_vertices-1,maximun_n_internal_vertices)
print("height without root: "+str(math.ceil(height_without_root)))

Calculating height
log base pointers_per_internal_vertices (number_of_records / ((pointers_per_root_vertices + 1) * records_per_leaf_vertices))
= log base 511.0 (1440000000 / ((512.0 + 1) * 292.0))
= log base 511.0 (9613.073780341265) = 1.4705477554603545
Rounded up: ceil(1.4705477554603545) = 2
height without root: 2


6. We build a secondary B+tree index on (id, time stamp). Assume leaf nodes are 1 block buckets that
contain a sequence of timestamps and include pointers to the next and previous buckets. Note that
the records associated with these timestamps could be accessed via the primary index. Assume these
buckets are completely full. What is the maximum height of such an secondary index?

# breakdown 
1. Build new B+-tree
2. it uses index on(id,timestamp)

Assume leaf nodes are **1 block** bucket
- sequence of timestamps 
- pointer to next and previous 

records associated with **these timestamps** could be accessed via the primary key

bucket are completely full

**Question:** what is the maximum height of such secondary index 

In [7]:
# id are the keys 
# leaf noes = timestamp 

bucket = SIZE_DISK_BLOCK
bucket -= 2 * SIZE_PTR

key_amount = bucket/SIZE_TIMESTAMP

ptr_n_per_vertices = internal_n_per_vertices(SIZE_DISK_BLOCK,SIZE_TIMESTAMP + SIZE_ID_MEASUREMENT,SIZE_PTR)

get_height(records,math.ceil(key_amount/2),math.floor((ptr_n_per_vertices -1)/2),1)





calculating internal ptr per vertices
(Page_size + Key_size)/(Ptr_size + Key_size) = (8192 + 12)/(8 + 12)
internal pointers per vertices are: 410.0

Calculating height
log base pointers_per_internal_vertices (number_of_records / ((pointers_per_root_vertices + 1) * records_per_leaf_vertices))
= log base 204 (1440000000 / ((1 + 1) * 511))
= log base 204 (1409001.9569471625) = 2.662292726782993
Rounded up: ceil(2.662292726782993) = 3


3

7. In words what would the Index of 6 let you query for/do?

**Answer**: make it easier to track individuals 

8. Could this be implemented on a single server with a 100 Mb/s network connection? (assume that
packets have minimal overhead and they are distributed randomly over a one second interval, there are
no confirmation receipts, each device has a separate IP)

In [8]:
people = 10*1000

data = SIZE_RECORD * people

print(f"The data that is transferred going to be transferred: {data*8}")

transfer_speed = 100 * pow(10,6)
print(f"The transfer speed: {transfer_speed}")

print(f"compare transfer_speed > data = {transfer_speed} > {data * 8} = {transfer_speed > data *8}")

The data that is transferred going to be transferred: 2240000
The transfer speed: 100000000
compare transfer_speed > data = 100000000 > 2240000 = True


**setup connection**